# MCD per-class height distributions

Loads the JSON produced by `collect_class_heights_mcd.py` and plots the
distribution of height-above-per-scan-bottom for each common class.
Heights are measured along the first scan's lidar +z axis, exactly as the
OSM height filter bins them at runtime.

In [ ]:
import json
import os

import numpy as np
import matplotlib.pyplot as plt

JSON_PATH = os.path.join(os.getcwd(), 'class_heights_mcd.json')
with open(JSON_PATH) as f:
    data = json.load(f)

hist_cfg = data['histogram']
edges = np.asarray(hist_cfg['bin_edges_m'], dtype=np.float64)
centers = 0.5 * (edges[:-1] + edges[1:])
step = hist_cfg['step_m']

print(f"dataset={data['dataset']}  sequence={data.get('sequence_name')}  "
      f"scans={data['num_scans']}  up_ref={data['lidar_up_ref']}")
print(f"histogram: {hist_cfg['num_bins']} bins x {step} m up to {hist_cfg['max_m']} m")

In [ ]:
classes = data['classes']
rows = []
for name, s in classes.items():
    rows.append((name, s['count'], s['mean'], s['median'], s['std'], s['min'], s['max']))

print(f"{'class':<12} {'count':>10} {'mean':>8} {'median':>8} {'std':>8} {'min':>8} {'max':>8}")
for name, c, mean, med, std, mn, mx in rows:
    if c == 0:
        print(f"{name:<12} {c:>10}")
        continue
    print(f"{name:<12} {c:>10} {mean:>8.2f} {(med or 0):>8.2f} {std:>8.2f} {mn:>8.2f} {mx:>8.2f}")

## Per-class histogram grid
Each subplot shows the height-above-bottom distribution for one class.
The dashed line marks the class mean.

In [ ]:
non_empty = [(name, s) for name, s in classes.items() if s['count'] > 0]
n = len(non_empty)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.0 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()

for ax, (name, s) in zip(axes, non_empty):
    h = np.asarray(s['histogram'], dtype=np.float64)
    total = h.sum()
    density = h / total if total > 0 else h
    ax.bar(centers, density, width=step, align='center', color='#3b82f6', edgecolor='none')
    if s['mean'] is not None:
        ax.axvline(s['mean'], color='#ef4444', ls='--', lw=1.5, label=f"mean={s['mean']:.2f} m")
    if s['median'] is not None:
        ax.axvline(s['median'], color='#10b981', ls=':', lw=1.5, label=f"median={s['median']:.2f} m")
    ax.set_title(f"{name}  (n={s['count']:,})")
    ax.set_xlabel('height above bottom (m)')
    ax.set_ylabel('fraction')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.2)

for ax in axes[len(non_empty):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## Overlay: all classes on one axis
Normalized densities so shapes are comparable across classes with very
different point counts.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap('tab10')
for i, (name, s) in enumerate(non_empty):
    h = np.asarray(s['histogram'], dtype=np.float64)
    total = h.sum()
    if total <= 0:
        continue
    ax.plot(centers, h / total, lw=1.6, color=cmap(i % 10), label=name)

ax.set_xlabel('height above bottom (m)')
ax.set_ylabel('fraction of class points')
ax.set_title('MCD per-class height-above-bottom distributions')
ax.grid(alpha=0.2)
ax.legend(loc='upper right', ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

## Mean height bar chart

In [ ]:
names = [n for n, _ in non_empty]
means = [s['mean'] for _, s in non_empty]
stds = [s['std'] for _, s in non_empty]

fig, ax = plt.subplots(figsize=(8, 4))
xs = np.arange(len(names))
ax.bar(xs, means, yerr=stds, color='#3b82f6', alpha=0.85, capsize=4)
ax.set_xticks(xs)
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylabel('mean height above bottom (m)')
ax.set_title('Per-class mean height  (error bars = std)')
ax.grid(alpha=0.2, axis='y')
plt.tight_layout()
plt.show()